## removing polygons with buildings

In [ ]:
import ast
import pandas as pd
from shapely.geometry import Polygon
from google.cloud import bigquery
from google.oauth2 import service_account
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==================================
# BIGQUERY CONFIG
# ==================================

key_path = "google_project_key.json"
project_id = "duality-extended"

credentials = service_account.Credentials.from_service_account_file(
    key_path
)

client = bigquery.Client(
    credentials=credentials,
    project=project_id
)

# ==================================
# LOAD DATA
# ==================================

df = pd.read_csv("info.csv")

# ==================================
# BUILDING CHECK FUNCTION
# ==================================

def has_building(polyinfo):

    try:

        data = ast.literal_eval(polyinfo)

        polygon = Polygon(data["coords"])

        polygon_wkt = polygon.wkt

        query = f"""
        SELECT COUNT(*) AS building_count
        FROM `bigquery-public-data.overture_maps.building`
        WHERE ST_INTERSECTS(
            geometry,
            ST_GEOGFROMTEXT('{polygon_wkt}')
        )
        """

        result = client.query(query).to_dataframe()

        return result.iloc[0]["building_count"] > 0

    except Exception as e:

        print(f"Error: {e}")

        return None

# ==================================
# PARALLEL PROCESSING
# ==================================

results = [None] * len(df)

with ThreadPoolExecutor(max_workers=50) as executor:

    future_to_index = {
        executor.submit(
            has_building,
            row["polyinfo"]
        ): idx

        for idx, row in df.iterrows()
    }

    completed = 0

    for future in as_completed(future_to_index):

        idx = future_to_index[future]

        results[idx] = future.result()

        completed += 1

        if completed % 25 == 0:
            print(
                f"Processed {completed}/{len(df)}"
            )

# ==================================
# SAVE RESULTS
# ==================================

df["has_building"] = results

print("\nBuilding Summary")
print(df["has_building"].value_counts(dropna=False))


# ==================================
# FARMS WITHOUT BUILDINGS
# ==================================

df_no_building = df[
    df["has_building"] == False
].copy()

df_no_building = df_no_building.drop(columns='has_building')


df_no_building.to_csv(
    "new_info.csv",
    index=False
)

print(
    f"\nOriginal farms: {len(df)}"
)

print(
    f"Farms without buildings: {len(df_no_building)}"
)

print(
    "Saved new_info.csv"
)

/Users/parthbansal/Satyukt Analytics/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Processed 25/854
Processed 50/854
Processed 75/854
Processed 100/854
Processed 125/854
Processed 150/854
Processed 175/854
Processed 200/854
Processed 225/854
Processed 250/854
Processed 275/854
Processed 300/854
Processed 325/854
Processed 350/854
Processed 375/854
Processed 400/854
Processed 425/854
Processed 450/854
Processed 475/854
Processed 500/854
Processed 525/854
Processed 550/854
Processed 575/854
Processed 600/854
Processed 625/854
Processed 650/854
Processed 675/854
Processed 700/854
Processed 725/854
Processed 750/854
Processed 775/854
Processed 800/854
Processed 825/854
Processed 850/854

Building Summary
has_building
False    702
True     152
Name: count, dtype: int64

Original farms: 854
Farms without buildings: 702
Saved new_info1.csv


# Predicting Heatwave

## Fetching 3 day temperatures for info.csv

In [ ]:
import pandas as pd
import numpy as np
import ast
import requests
from datetime import timedelta

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv("new_info.csv")
df["upload_time"] = pd.to_datetime(df["upload_time"])

# ==========================================
# CACHES
# ==========================================

weather_cache = {}
normal_temp_cache = {}

# ==========================================
# HELPERS
# ==========================================

def round_to_quarter(value):
    return round(round(value / 0.25) * 0.25, 2)


def get_centroid(poly_string):

    data = ast.literal_eval(poly_string)

    coords = data["coords"]

    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]

    center_lon = sum(lons) / len(lons)
    center_lat = sum(lats) / len(lats)

    return (
        data.get("name"),
        center_lat,
        center_lon
    )

# ==========================================
# EXTRACT CENTROIDS
# ==========================================

farm_names = []
latitudes = []
longitudes = []

for poly in df["polyinfo"]:

    name, lat, lon = get_centroid(poly)

    farm_names.append(name)
    latitudes.append(lat)
    longitudes.append(lon)

df["farm_name"] = farm_names
df["latitude"] = latitudes
df["longitude"] = longitudes

df["grid_lat"] = df["latitude"].apply(
    round_to_quarter
)

df["grid_lon"] = df["longitude"].apply(
    round_to_quarter
)

df["date"] = df["upload_time"].dt.date
df["month"] = df["upload_time"].dt.month

# ==========================================
# MONTHLY NORMAL CACHE
# ==========================================

def get_normal_temp(lat, lon, month):

    key = (lat, lon, month)

    if key in normal_temp_cache:
        return normal_temp_cache[key]

    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}"
        f"&longitude={lon}"
        f"&start_date=2024-{month:02d}-01"
        f"&end_date=2024-{month:02d}-28"
        "&daily=temperature_2m_mean"
        "&timezone=auto"
    )

    response = requests.get(
        url,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    normal_temp = round(
        np.mean(
            data["daily"][
                "temperature_2m_mean"
            ]
        ),
        2
    )

    normal_temp_cache[key] = normal_temp

    return normal_temp

# ==========================================
# UNIQUE WEATHER REQUESTS
# ==========================================

unique_weather = (
    df[
        [
            "grid_lat",
            "grid_lon",
            "date",
            "month"
        ]
    ]
    .drop_duplicates()
)

feature_cache = {}

# ==========================================
# WEATHER FETCHING
# ==========================================

for _, row in unique_weather.iterrows():

    lat = row["grid_lat"]
    lon = row["grid_lon"]

    target_date = pd.Timestamp(
        row["date"]
    )

    month = row["month"]

    weather_key = (
        lat,
        lon,
        target_date.date()
    )

    if weather_key in weather_cache:
        feature_cache[
            weather_key
        ] = weather_cache[
            weather_key
        ]
        continue

    try:

        start_date = (
            target_date -
            timedelta(days=3)
        ).strftime("%Y-%m-%d")

        end_date = target_date.strftime(
            "%Y-%m-%d"
        )

        url = (
            "https://archive-api.open-meteo.com/v1/archive?"
            f"latitude={lat}"
            f"&longitude={lon}"
            f"&start_date={start_date}"
            f"&end_date={end_date}"
            "&daily=temperature_2m_max"
            "&timezone=auto"
        )

        response = requests.get(
            url,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        temps = data["daily"][
            "temperature_2m_max"
        ]

        if len(temps) < 4:
            continue

        tmax_prev3 = temps[0]
        tmax_prev2 = temps[1]
        tmax_prev1 = temps[2]
        tmax_today = temps[3]

        rolling_values = [
            tmax_prev3,
            tmax_prev2,
            tmax_prev1
        ]

        normal_temp = get_normal_temp(
            lat,
            lon,
            month
        )

        features = {

            "tmax_prev1":
                tmax_prev1,

            "tmax_prev2":
                tmax_prev2,

            "tmax_prev3":
                tmax_prev3,

            "rolling_3day_avg":
                round(
                    np.mean(
                        rolling_values
                    ),
                    2
                ),

            "rolling_3day_max":
                round(
                    np.max(
                        rolling_values
                    ),
                    2
                ),

            "rolling_3day_std":
                round(
                    np.std(
                        rolling_values
                    ),
                    2
                ),

            "temp_change_3day":
                round(
                    tmax_today -
                    tmax_prev3,
                    2
                ),
            "normal_temp":
                normal_temp,

            "departure":
                round(
                    tmax_today -
                    normal_temp,
                    2
                )
        }

        weather_cache[
            weather_key
        ] = features

        feature_cache[
            weather_key
        ] = features

    except Exception as e:

        print(
            f"Error: {lat}, {lon}, "
            f"{target_date.date()} -> {e}"
        )

# ==========================================
# MAP FEATURES BACK
# ==========================================

features_list = []

for _, row in df.iterrows():

    key = (
        row["grid_lat"],
        row["grid_lon"],
        row["date"]
    )

    features_list.append(
        feature_cache.get(
            key,
            {}
        )
    )

feature_df = pd.DataFrame(
    features_list
)

df = pd.concat(
    [
        df.reset_index(drop=True),
        feature_df
    ],
    axis=1
)

# ==========================================
# REMOVE TEMP COLUMNS
# ==========================================

df.drop(
    columns=[
        "grid_lat",
        "grid_lon",
        "date",
        "month"
    ],
    inplace=True
)

# ==========================================
# SAVE
# ==========================================

df.to_csv(
    "farm_heatwave_features.csv",
    index=False
)

print(
    f"Rows: {len(df)}"
)

print(
    f"Unique weather requests: "
    f"{len(weather_cache)}"
)

print(
    f"Unique normal-temp requests: "
    f"{len(normal_temp_cache)}"
)

print(
    "Saved farm_heatwave_features.csv"
)

## Model Prediction

In [ ]:
import joblib
import pandas as pd

model = joblib.load(
    "severe_heatwave_model2.pkl"
)

threshold = 0.30


farm_df = pd.read_csv("farm_heatwave_features.csv")

farm_features = [
    'tmax_prev1',
    'tmax_prev2',
    'tmax_prev3',
    'rolling_3day_avg',
    'rolling_3day_max',                                                                                 
    'rolling_3day_std',
    'temp_change_3day',
    'departure'
]

farm_df = farm_df.dropna(
    subset=farm_features
)

farm_probabilities = model.predict_proba(
    farm_df[farm_features]
)[:, 1]

farm_predictions = (
    farm_probabilities > threshold
).astype(int)

farm_df['severe_heatwave_probability'] = (
    farm_probabilities * 100
)

farm_df['severe_heatwave_prediction'] = (
    farm_predictions
)


farm_df.to_csv("farm_heatwave_predictions.csv",index=False)

## Fetching temp values at specific lat and lon

In [ ]:
import pandas as pd

# Load predictions CSV
df = pd.read_csv("farm_heatwave_predictions.csv")

target_lat = 18.407693176045175
target_lon = 77.67428451695235

# Distance squared
df["distance"] = (
    (df["latitude"] - target_lat) ** 2 +
    (df["longitude"] - target_lon) ** 2
)

# Nearest row
nearest = df.loc[
    df["distance"].idxmin()
]

print(
    nearest[
        [
            "farm_id",
            "farm_name",
            "latitude",
            "longitude",
            "upload_time",
            "tmax_prev1",
            "normal_temp",
            "departure",
            "severe_heatwave_probability",
            "severe_heatwave_prediction"
        ]
    ]
)